In [7]:
!pip install -q tqdm librosa

from pathlib import Path
import json, subprocess, warnings
import numpy as np
import pandas as pd
import librosa
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")


In [8]:
from google.colab import files

print("UPLOAD YOUR EXISTING split_manifest.csv")
uploaded = files.upload()

names = list(uploaded.keys())
if "split_manifest.csv" in names:
    manifest_name = "split_manifest.csv"
elif len(names) == 1 and names[0].lower().endswith(".csv"):
    manifest_name = names[0]
else:
    raise ValueError("Please upload exactly your existing split_manifest.csv.")

MANIFEST_PATH = Path("/content") / manifest_name
print("Using:", MANIFEST_PATH)


UPLOAD YOUR EXISTING split_manifest.csv


Saving split_manifest.csv to split_manifest (1).csv
Using: /content/split_manifest (1).csv


In [9]:
manifest = pd.read_csv(MANIFEST_PATH)

expected = ["relative_path", "genre", "split"]
if list(manifest.columns) != expected:
    raise ValueError(f"Expected columns {expected}, got {list(manifest.columns)}")

if len(manifest) != 1000:
    raise ValueError(f"Expected 1000 rows, got {len(manifest)}")

if manifest["relative_path"].duplicated().any():
    raise ValueError("Duplicate relative_path values found.")

expected_splits = {"train":700, "validation":150, "test":150}
actual_splits = manifest["split"].value_counts().to_dict()
if actual_splits != expected_splits:
    raise ValueError(f"Wrong split counts: {actual_splits}")

genres = manifest["genre"].value_counts()
if len(genres) != 10 or not (genres == 100).all():
    raise ValueError("Expected 10 genres with 100 tracks each.")

print("MANIFEST VALIDATED")
print("Tracks:", len(manifest))
print("Splits:", actual_splits)
print("Genres:", sorted(manifest["genre"].unique()))
display(manifest.head())


MANIFEST VALIDATED
Tracks: 1000
Splits: {'train': 700, 'test': 150, 'validation': 150}
Genres: ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']


,relative_path,genre,split
0,genres/blues/blues.00001.au,blues,test
1,genres/blues/blues.00021.au,blues,test
2,genres/blues/blues.00029.au,blues,test
3,genres/blues/blues.00032.au,blues,test
4,genres/blues/blues.00037.au,blues,test


In [10]:
# Automatically find or download GTZAN

GTZAN_URL = (
    "https://ibm.ent.box.com/index.php?"
    "rm=box_download_shared_file&"
    "shared_name=gvkgmb4n0h6rbeccdwtwa0ujjjfmouof&"
    "file_id=f_961929491012"
)

work = Path("/content/gtzan_work")
archive = work / "gtzan_archive"
extract = work / "extracted"
extract.mkdir(parents=True, exist_ok=True)

first = Path(manifest.iloc[0]["relative_path"])
audio_base = None

for root in [Path("/content"), Path("/content/music-genre-classification")]:
    if (root / first).exists():
        audio_base = root.resolve()
        break

if audio_base is None:
    print("GTZAN audio not found. Downloading automatically (~1.2 GB)...")
    work.mkdir(parents=True, exist_ok=True)
    if not archive.exists() or archive.stat().st_size < 100_000_000:
        subprocess.run(["wget","-c","--show-progress","-O",str(archive),GTZAN_URL], check=True)

    ftype = subprocess.run(["file","-b",str(archive)],capture_output=True,text=True,check=True).stdout.lower()
    print("Archive:", ftype.strip())

    if "zip archive" in ftype:
        subprocess.run(["unzip","-q","-o",str(archive),"-d",str(extract)],check=True)
    elif "gzip compressed" in ftype:
        subprocess.run(["tar","-xzf",str(archive),"-C",str(extract)],check=True)
    elif "tar archive" in ftype:
        subprocess.run(["tar","-xf",str(archive),"-C",str(extract)],check=True)
    else:
        raise RuntimeError("Unknown GTZAN archive format.")

    found = list(extract.rglob(first.name))
    if not found:
        raise FileNotFoundError(f"Could not find {first.name} after extraction.")
    audio_base = found[0].parents[2].resolve()

print("Audio base:", audio_base)


GTZAN audio not found. Downloading automatically (~1.2 GB)...
Archive: gzip compressed data, was "genres.tar", last modified: fri may 22 19:53:15 2009, from unix, original size modulo 2^32 1325035520
Audio base: /content/gtzan_work/extracted


In [11]:
manifest["audio_path"] = manifest["relative_path"].map(lambda p: audio_base / p)
missing = manifest.loc[~manifest["audio_path"].map(Path.exists)]

print("Expected:", len(manifest))
print("Found:", len(manifest)-len(missing))
print("Missing:", len(missing))

if len(missing):
    display(missing[["relative_path","genre","split"]].head(20))
    raise FileNotFoundError(f"{len(missing)} audio files are missing.")

print("SUCCESS — all 1000 audio files found.")


Expected: 1000
Found: 1000
Missing: 0
SUCCESS — all 1000 audio files found.


In [12]:
N_MFCC, N_FFT, HOP = 13, 2048, 512

def ms(x):
    x = np.asarray(x, dtype=np.float32)
    return np.r_[x.mean(axis=1), x.std(axis=1)]

def extract(path):
    y, sr = librosa.load(path, sr=None, mono=True)
    mfcc = librosa.feature.mfcc(y=y,sr=sr,n_mfcc=N_MFCC,n_fft=N_FFT,hop_length=HOP)
    delta = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)
    chroma = librosa.feature.chroma_stft(y=y,sr=sr,n_fft=N_FFT,hop_length=HOP)
    centroid = librosa.feature.spectral_centroid(y=y,sr=sr,n_fft=N_FFT,hop_length=HOP)
    bandwidth = librosa.feature.spectral_bandwidth(y=y,sr=sr,n_fft=N_FFT,hop_length=HOP)
    rolloff = librosa.feature.spectral_rolloff(y=y,sr=sr,n_fft=N_FFT,hop_length=HOP)
    contrast = librosa.feature.spectral_contrast(y=y,sr=sr,n_fft=N_FFT,hop_length=HOP)
    rms = librosa.feature.rms(y=y,frame_length=N_FFT,hop_length=HOP)
    zcr = librosa.feature.zero_crossing_rate(y,frame_length=N_FFT,hop_length=HOP)
    tempo = librosa.feature.tempo(y=y,sr=sr,hop_length=HOP)[0]

    f = np.r_[ms(mfcc),ms(delta),ms(delta2),ms(chroma),
              ms(centroid),ms(bandwidth),ms(rolloff),ms(contrast),
              ms(rms),ms(zcr),tempo]
    if len(f) != 137:
        raise ValueError(f"Expected 137 features, got {len(f)}")
    return f.astype(np.float32)


In [14]:
N_MFCC, N_FFT, HOP = 13, 2048, 512

def ms(x):
    x = np.asarray(x, dtype=np.float32)
    return np.r_[x.mean(axis=1), x.std(axis=1)]


def extract_features(path):
    y, sr = librosa.load(
        path,
        sr=None,
        mono=True
    )

    # --------------------------------------------------
    # 1. MFCC
    # --------------------------------------------------
    mfcc = librosa.feature.mfcc(
        y=y,
        sr=sr,
        n_mfcc=N_MFCC,
        n_fft=N_FFT,
        hop_length=HOP
    )

    delta = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)

    # --------------------------------------------------
    # 2. Chroma
    # --------------------------------------------------
    chroma = librosa.feature.chroma_stft(
        y=y,
        sr=sr,
        n_fft=N_FFT,
        hop_length=HOP
    )

    # --------------------------------------------------
    # 3. Spectral features
    # --------------------------------------------------
    centroid = librosa.feature.spectral_centroid(
        y=y,
        sr=sr,
        n_fft=N_FFT,
        hop_length=HOP
    )

    bandwidth = librosa.feature.spectral_bandwidth(
        y=y,
        sr=sr,
        n_fft=N_FFT,
        hop_length=HOP
    )

    rolloff = librosa.feature.spectral_rolloff(
        y=y,
        sr=sr,
        n_fft=N_FFT,
        hop_length=HOP
    )

    contrast = librosa.feature.spectral_contrast(
        y=y,
        sr=sr,
        n_fft=N_FFT,
        hop_length=HOP
    )

    # Additional spectral descriptors
    flatness = librosa.feature.spectral_flatness(
        y=y,
        n_fft=N_FFT,
        hop_length=HOP
    )

    # --------------------------------------------------
    # 4. Spectral flux
    # --------------------------------------------------
    stft = np.abs(
        librosa.stft(
            y,
            n_fft=N_FFT,
            hop_length=HOP
        )
    )

    stft_norm = stft / (
        np.sum(stft, axis=0, keepdims=True) + 1e-10
    )

    flux = np.sqrt(
        np.sum(
            np.diff(stft_norm, axis=1) ** 2,
            axis=0
        )
    )

    # Pad first frame so dimensions align
    flux = np.concatenate([[0.0], flux])[np.newaxis, :]

    # --------------------------------------------------
    # 5. Spectral entropy
    # --------------------------------------------------
    power = stft ** 2

    probability = power / (
        np.sum(power, axis=0, keepdims=True) + 1e-10
    )

    entropy = -np.sum(
        probability * np.log2(probability + 1e-10),
        axis=0
    )

    entropy = entropy[np.newaxis, :]

    # --------------------------------------------------
    # 6. Spectral skewness
    # --------------------------------------------------
    freq_indices = np.arange(stft.shape[0], dtype=np.float32)[:, np.newaxis]

    weighted_mean = np.sum(
        freq_indices * stft_norm,
        axis=0
    )

    variance = np.sum(
        ((freq_indices - weighted_mean) ** 2) * stft_norm,
        axis=0
    )

    std = np.sqrt(variance + 1e-10)

    skewness = np.sum(
        ((freq_indices - weighted_mean) / std) ** 3
        * stft_norm,
        axis=0
    )

    skewness = skewness[np.newaxis, :]

    # --------------------------------------------------
    # 7. Spectral kurtosis
    # --------------------------------------------------
    kurtosis = np.sum(
        ((freq_indices - weighted_mean) / std) ** 4
        * stft_norm,
        axis=0
    )

    kurtosis = kurtosis[np.newaxis, :]

    # --------------------------------------------------
    # 8. RMS energy
    # --------------------------------------------------
    rms = librosa.feature.rms(
        y=y,
        frame_length=N_FFT,
        hop_length=HOP
    )

    # --------------------------------------------------
    # 9. Zero crossing rate
    # --------------------------------------------------
    zcr = librosa.feature.zero_crossing_rate(
        y,
        frame_length=N_FFT,
        hop_length=HOP
    )

    # --------------------------------------------------
    # 10. Tempo
    # --------------------------------------------------
    tempo = librosa.feature.tempo(
        y=y,
        sr=sr,
        hop_length=HOP
    )[0]

    # --------------------------------------------------
    # Combine everything
    # --------------------------------------------------
    features = np.r_[
        ms(mfcc),          # 26
        ms(delta),         # 26
        ms(delta2),        # 26
        ms(chroma),        # 24

        ms(centroid),      # 2
        ms(bandwidth),     # 2
        ms(rolloff),       # 2
        ms(contrast),      # 14

        ms(flatness),      # 2
        ms(flux),           # 2
        ms(entropy),        # 2
        ms(skewness),       # 2
        ms(kurtosis),       # 2

        ms(rms),            # 2
        ms(zcr),            # 2

        tempo               # 1
    ]

    if len(features) != 137:
        raise ValueError(
            f"Expected 137 features, got {len(features)}"
        )

    return features.astype(np.float32)

In [18]:
feature_names = []

# ==========================================================
# 1. MFCC
# ==========================================================
for i in range(1, 14):
    feature_names.append(f"mfcc_{i}_mean")
for i in range(1, 14):
    feature_names.append(f"mfcc_{i}_std")


# ==========================================================
# 2. Delta MFCC
# ==========================================================
for i in range(1, 14):
    feature_names.append(f"delta_mfcc_{i}_mean")
for i in range(1, 14):
    feature_names.append(f"delta_mfcc_{i}_std")


# ==========================================================
# 3. Delta-Delta MFCC
# ==========================================================
for i in range(1, 14):
    feature_names.append(f"delta2_mfcc_{i}_mean")
for i in range(1, 14):
    feature_names.append(f"delta2_mfcc_{i}_std")


# ==========================================================
# 4. Chroma
# ==========================================================
for i in range(1, 13):
    feature_names.append(f"chroma_{i}_mean")
for i in range(1, 13):
    feature_names.append(f"chroma_{i}_std")


# ==========================================================
# 5. Basic spectral features
# ==========================================================
for name in [
    "spectral_centroid",
    "spectral_bandwidth",
    "spectral_rolloff"
]:
    feature_names.append(f"{name}_mean")
    feature_names.append(f"{name}_std")


# ==========================================================
# 6. Spectral contrast
# ==========================================================
for i in range(1, 8):
    feature_names.append(f"spectral_contrast_{i}_mean")
    feature_names.append(f"spectral_contrast_{i}_std")


# ==========================================================
# 7. Additional spectral features
# ==========================================================
for name in [
    "spectral_flatness",
    "spectral_flux",
    "spectral_entropy",
    "spectral_skewness",
    "spectral_kurtosis"
]:
    feature_names.append(f"{name}_mean")
    feature_names.append(f"{name}_std")


# ==========================================================
# 8. Energy / Rhythm
# ==========================================================
feature_names += [
    "rms_mean",
    "rms_std",
    "zcr_mean",
    "zcr_std",
    "tempo"
]


# ==========================================================
# VALIDATION
# ==========================================================
print("Total feature names:", len(feature_names))

assert len(feature_names) == 137

print("137 feature names ready.")

Total feature names: 137
137 feature names ready.


In [19]:
records = []
errors = []

for row in tqdm(
    manifest.itertuples(index=False),
    total=len(manifest),
    desc="Extracting features"
):
    try:
        # IMPORTANT:
        # Use the corrected 137-feature function
        r = extract_features(Path(row.audio_path))

        record = {
            "relative_path": row.relative_path,
            "genre": row.genre,
            "split": row.split
        }

        record.update(dict(zip(feature_names, r)))
        records.append(record)

    except Exception as e:
        errors.append({
            "relative_path": row.relative_path,
            "genre": row.genre,
            "error": repr(e)
        })

if errors:
    error_df = pd.DataFrame(errors)

    print(f"Files failed: {len(error_df)}")
    display(error_df.head(20))

    raise RuntimeError(
        f"{len(error_df)} files failed."
    )

features_df = pd.DataFrame(records)

print("======================================")
print("FEATURE EXTRACTION COMPLETE")
print("======================================")
print("Rows:", len(features_df))
print("Columns:", len(features_df.columns))
print("======================================")

Extracting features:   0%|          | 0/1000 [00:00<?, ?it/s]

FEATURE EXTRACTION COMPLETE
Rows: 1000
Columns: 140


In [20]:
meta = ["relative_path","genre","split"]
num = [c for c in features_df.columns if c not in meta]

assert features_df.shape == (1000,140)
assert len(num) == 137
assert features_df["relative_path"].is_unique
assert np.isfinite(features_df[num].to_numpy(dtype=float)).all()

out = Path("/content/handcrafted_output")
out.mkdir(exist_ok=True)

csv_path = out / "handcrafted_features.csv"
json_path = out / "handcrafted_feature_config.json"

features_df.to_csv(csv_path,index=False)

with open(json_path,"w") as f:
    json.dump({
        "dataset":"GTZAN",
        "tracks":1000,
        "numeric_features":137,
        "output_shape":[1000,140],
        "feature_names":feature_names,
        "split_policy":"existing split_manifest.csv; no new split"
    },f,indent=2)

print("======================================")
print("HANDCRAFTED FEATURES COMPLETE")
print("CSV:",csv_path)
print("Shape:",features_df.shape)
print("======================================")


HANDCRAFTED FEATURES COMPLETE
CSV: /content/handcrafted_output/handcrafted_features.csv
Shape: (1000, 140)


In [21]:
from google.colab import files

files.download(
    "/content/handcrafted_output/handcrafted_features.csv"
)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [22]:
files.download(
    "/content/handcrafted_output/handcrafted_feature_config.json"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>